# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata: use the Dataset object properties
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field `@id`s
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (@id):")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, dataType: {field.data_type})")
    print()

## Example records per record set
Let's inspect a few records from the first available record set.

In [ ]:
if len(record_sets) == 0:
    print("No record sets found in the dataset.")
else:
    example_record_set = record_sets[0].id
    print(f"Example records for record set @id: {example_record_set}\n")
    for i, rec in enumerate(dataset.records(record_set=example_record_set)):
        print(json.dumps(rec, indent=2))
        if i >= 2:
            break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis.

Use record set and field `@id`s from the overview step above.

In [ ]:
# Store all dataframes in a dictionary for convenience
dataframes = {}

for rs in record_sets:
    rs_id = rs.id
    # List of records for the record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df
    print(f"---\nRecord set: {rs_id}\nDataFrame columns: {list(df.columns)}. Sample records:")
    print(df.head(3))
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section will perform sample filtering, normalization, and grouping on an example numeric field.

In [ ]:
# For illustration, let's pick a record set with data
# You may change the selected record set or fields to match your analysis needs
import numpy as np

if len(record_sets) == 0:
    print("No record sets to analyze.")
else:
    # User should change these based on available columns from earlier cells, e.g.:
    selected_record_set_id = record_sets[0].id
    df = dataframes[selected_record_set_id]
    print(f"Working with record set: {selected_record_set_id}")
    if df.empty:
        print("No data available in this record set.")
    else:
        print(f"Available columns: {list(df.columns)}")

        # Try to auto-select a numeric field by dtype
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
        print(f"Numeric fields detected: {numeric_fields}")

        if len(numeric_fields) == 0:
            print("No numeric columns available for filtering and normalization.")
        else:
            numeric_field = numeric_fields[0]
            threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
            # Filter records
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            print(filtered_df[[numeric_field]].head())

            # Normalize numeric field
            if filtered_df[numeric_field].std() != 0:
                filtered_df[f"{numeric_field}_normalized"] = (
                    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
                )
                print(f"Normalized {numeric_field} for filtered records:")
                print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])
            else:
                print(f"No variance in '{numeric_field}'; skipping normalization.")

            # Attempt to group by the first non-numeric field
            non_numeric_fields = [c for c in df.columns if c not in numeric_fields]
            if len(non_numeric_fields) > 0:
                group_field = non_numeric_fields[0]
                group_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by '{group_field}':")
                print(group_df.head())
            else:
                print("No non-numeric fields available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and not df.empty and len(numeric_fields) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, ax=ax)
    ax.set_title(f"Distribution of '{numeric_field}' in '{selected_record_set_id}'")
    ax.set_xlabel(numeric_field)
    ax.set_ylabel('Count')
    plt.show()

    if len(non_numeric_fields) > 0:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[non_numeric_fields[0]], y=df[numeric_field])
        plt.title(f"{numeric_field} by {non_numeric_fields[0]}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and performed simple processing of the FAIR² dataset using the `mlcroissant` library. We demonstrated how to examine available record sets and fields by their `@id`, extract records into DataFrames for analysis, and apply basic numerical filtering, normalization, grouping, and visualization. 

For further analysis, refer to the documentation for more advanced data processing and utilize field or column `@id`s as needed. If you encounter issues with field detection or require domain-specific EDA, consult the FAIR² schema definitions for exact field names and descriptions.